In [25]:
from pyspark.sql import SparkSession
import pandas

spark = SparkSession.builder \
    .appName("casestudy-dem1") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Started")

Spark Started


In [26]:
customers_df = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
customers_df.show(5)

order_items_df = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
order_items_df.show(5)

products_df = spark.read.csv("data/products.csv", header=True, inferSchema=True)
products_df.show(5)

returns_df = spark.read.csv("data/returns.csv", header=True, inferSchema=True)
returns_df.show(5)

orders_df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
orders_df.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows



+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|       2|       203.02|
|            3|  426242|      2818|       5|      1061.81|
|            4|  236965|     35931|       5|      1005.49|
|            5|  289552|     48310|       4|       540.78|
+-------------+--------+----------+--------+-------------+
only showing top 5 rows

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electronics|Brand_A|   332.22|
|         3|   Product_3|        Sports|Brand_D|    68.58|
|         4|   Product_4|      Clothing|Brand_A|   729.19|
|         5|   Product_5|Home &

[Stage 164:>                                                        (0 + 2) / 2]

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|payment_mode|order_status|
+--------+-----------+----------+------------+------------+
|       1|      54630|2024-01-25| Credit Card|   Delivered|
|       2|      22415|2024-07-08| Credit Card|   Delivered|
|       3|      20909|2024-01-23|         UPI|   Delivered|
|       4|      98027|2024-03-02| Credit Card|   Cancelled|
|       5|      31332|2024-12-17|         UPI|   Cancelled|
+--------+-----------+----------+------------+------------+
only showing top 5 rows



In [27]:
customers_df.createOrReplaceTempView("customers")
order_items_df.createOrReplaceTempView("order_items")
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
returns_df.createOrReplaceTempView("returns")

In [28]:
sql_df1 = spark.sql("""
    SELECT 
        (SELECT count(customer_id) FROM customers) as total_customers,
        (SELECT count(product_id) FROM products) as total_products,
        (SELECT count(order_id) FROM orders) as total_orders,
        (SELECT count(order_item_id) FROM order_items) as total_order_items,
        (SELECT count(return_id) FROM returns) as total_returns
""")

sql_df1.show()

[Stage 169:>  (0 + 2) / 2][Stage 170:>  (0 + 0) / 1][Stage 172:>  (0 + 0) / 1]

+---------------+--------------+------------+-----------------+-------------+
|total_customers|total_products|total_orders|total_order_items|total_returns|
+---------------+--------------+------------+-----------------+-------------+
|         100000|         50000|     1000000|          3000000|       100000|
+---------------+--------------+------------+-----------------+-------------+



In [29]:
sql_df1.write.mode("overwrite").csv("output/result1/FIRST",header=True)

In [30]:
sql_df2=spark.sql("""
SELECT category as product_category,
sum(unit_cost) as total_sales_amount
from products
group by category  """)
sql_df2.show()

+----------------+------------------+
|product_category|total_sales_amount|
+----------------+------------------+
|  Home & Kitchen| 2901364.330000004|
|          Sports| 2853163.040000003|
|     Electronics|2864604.7399999946|
|        Clothing| 2841424.610000002|
|           Books|2853871.8500000075|
|          Beauty|2919388.7500000037|
|            Toys|2851913.1100000013|
+----------------+------------------+



In [31]:
sql_df2.write.mode("overwrite").csv("output/result2",header=True)

In [32]:
sql_df3 = spark.sql("""
    SELECT 
        c.customer_id, 
        c.customer_name, 
        ROUND(SUM(oi.quantity * oi.selling_price), 2) as total_purchase_amount
    FROM customers c
    JOIN orders o 
        ON c.customer_id = o.customer_id
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    -- Only count completed sales! You probably don't want cancelled orders in your revenue count.
    WHERE o.order_status = 'Delivered' 
    GROUP BY 
        c.customer_id, 
        c.customer_name
    ORDER BY 
        total_purchase_amount DESC
    LIMIT 10
""")

sql_df3.show()

+-----------+--------------+---------------------+
|customer_id| customer_name|total_purchase_amount|
+-----------+--------------+---------------------+
|      64560|Customer_64560|            119030.04|
|      65135|Customer_65135|            111137.38|
|      52275|Customer_52275|            108098.33|
|      28584|Customer_28584|            107848.24|
|      37277|Customer_37277|            107444.08|
|      17810|Customer_17810|            105355.56|
|      11201|Customer_11201|             104247.4|
|      33876|Customer_33876|             102933.8|
|      10188|Customer_10188|            100520.46|
|      97963|Customer_97963|             99773.07|
+-----------+--------------+---------------------+



In [36]:
sql_df3.write.mode("overwrite").csv("output/result3",header=True)

In [44]:
sql_df4 = spark.sql("""
SELECT 
    YEAR(o.order_date) AS year,
    MONTH(o.order_date) AS month,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE YEAR(o.order_date) = (
    SELECT MAX(YEAR(order_date)) FROM orders
)
GROUP BY YEAR(o.order_date), MONTH(o.order_date)
ORDER BY year, month
""")

sql_df4.show()


[Stage 266:>                                                        (0 + 2) / 2]

+----+-----+--------------+
|year|month|   total_sales|
+----+-----+--------------+
|2024|    1|4.4457777576E8|
|2024|    2| 4.153661442E8|
|2024|    3|4.4362824541E8|
|2024|    4|4.2782097434E8|
|2024|    5|4.4481061895E8|
|2024|    6|4.3170515406E8|
|2024|    7|4.4367051912E8|
|2024|    8|4.4109517702E8|
|2024|    9|4.3107152608E8|
|2024|   10|4.4136378931E8|
|2024|   11|4.3362336404E8|
|2024|   12|4.4271290835E8|
+----+-----+--------------+

